In [1]:
# pip install langchain_huggingface 
# !pip install langchain_community

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
import spacy
import re
import contractions
from textblob import TextBlob
import os

C:\Users\graj5\AppData\Local\Temp\ipykernel_13520\3781176576.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# open()

## 1. Load the document(.txt)

In [4]:
data = TextLoader('ragflow.txt').load()[0].page_content
data

"RAG PIPELINE\nStep 1:\nData Loader\n    -word\n    -pdf\n    .txt\n    How ? and which tool do we use for this ?\n    langchain(framework)\n    Interview Questions:\n    how you will extract text information from (pdf with text) ?\n        -langchain\n    pdf with image ?\n        -inside image text content,solution -> OCR model(paddle ocr)\n        -graph based image,solution -> (VLM)\n    OCR ?\n     -with help of this Paddle OCR model we can extract the text inside image.\n    VLM ?\n     -we can get summary or description about graph.\n    How will you handle if pdf is having both text and image content ?\n    hybrid \n    1) text -> langchain\n    2) image -> VLM or OCR depends on scenario\n\n    merge both both text (one from langchain document loader and second from VLM or OCR) before \n    splitting the documents into multiple parts.\nStep 2)\nText normalizati\n1)page numbers\n2)unwanted characters(removing emoji's,spl characters,punctuations)\n3)exapand the words (example => 

## 2. Text Normalization

### Converting all the characters into lower case

In [5]:
data = data.lower()

### Removing extra spaces

In [6]:
data = re.sub(r'\s{2,}','',data)

### Removing numbers like 1.,2.

In [7]:
data = re.sub(r'\b\d+\.\b','',data)
data

"rag pipeline\nstep 1:\ndata loader-word-pdf.txthow ? and which tool do we use for this ?langchain(framework)interview questions:how you will extract text information from (pdf with text) ?-langchainpdf with image ?-inside image text content,solution -> ocr model(paddle ocr)-graph based image,solution -> (vlm)ocr ?-with help of this paddle ocr model we can extract the text inside image.vlm ?-we can get summary or description about graph.how will you handle if pdf is having both text and image content ?hybrid1) text -> langchain2) image -> vlm or ocr depends on scenariomerge both both text (one from langchain document loader and second from vlm or ocr) beforesplitting the documents into multiple parts.\nstep 2)\ntext normalizati\n1)page numbers\n2)unwanted characters(removing emoji's,spl characters,punctuations)\n3)exapand the words (example => didn't) did not,example 2 asap\n4)correcting the words(not possible to correct all the words)\n.\n.\n10)\nnlp\n1)nltk(natural language toolkit)\

### Contractions

In [8]:
data = contractions.fix(data)

### Removing punctuations & spl characters

In [9]:
data = re.sub(r'[^0-9a-zA-Z\s]','',data)

### TextBlob

In [10]:
# values= TextBlob(data).correct()
# values

### Lemmatization

In [11]:
import spacy

nlp = spacy.load('en_core_web_sm')
tokens = nlp(data)

In [12]:
# for token in tokens:
    # print(token.lemma_,token.is_stop)
    # if not token.is_stop:
    #     print(token)
updated_tokens = [str(token.lemma_) for token in tokens if not token.is_stop]
data = ' '.join(updated_tokens)
# data
# print(type(updated_tokens[0]))
print(data)

rag pipeline 
 step 1 
 datum loaderwordpdftxthow   tool use langchainframeworkinterview questionshow extract text information pdf text langchainpdf image inside image text contentsolution   ocr modelpaddle ocrgraph base imagesolution   vlmocr help paddle ocr model extract text inside imagevlm summary description graphhow handle pdf have text image content hybrid1 text   langchain2 image   vlm ocr depend scenariomerge text langchain document loader second vlm ocr beforesplitte document multiple part 
 step 2 
 text normalizati 
 1page number 
 2unwante charactersremove emojisspl characterspunctuation 
 3exapand word example   notexample 2 soon possible 
 4correcte wordsnot possible correct word 


 10 
 nlp 
 1nltknatural language toolkit 
 2spacy 
 step 3 
 chunk   convert document multiple part 
   chunk 
 tool   langchain 
 step 4 
 embedding   convert chunk numbersvector 
 tool langchain 
 technique   transformersencoder architecture 
 step 5 
 vector db 
 tool langchain 
 db faiss

### Chunking(Converting doc -> chunks)

In [13]:
'''
major types of chunking
- sentence level chunking
- word level chunking
- fixed size chunking
chunk_size = 10
chunk 1 = >machine le
chunk 2 = > arning use
- chunk overlap
- recursive chunking
- semantic chunking
'''
# chunk_size = 150
# # 0 - 149
# # 150 - 299
# chunks = [data[index:index+chunk_size] for index in range(0,len(data),chunk_size)]
# # for index in range(0,len(data),chunk_size):
# #     print(data[index:index+chunk_size])
# chunks

''' .

machine learning uses support vector machines support supply chain optimization improving loss functions modern systems 
 sentiment analysis deep learning benefits k nearest neighbors strengthens activation functions improves decision quality 
 
 
 neural networks applies naive bayes document summarization data augmentation helps model generalize effectively 
 practical study natural language processing shows principal component analysis improve video understanding refining class imbalance training 
 
 
 researchers use computer vision convolutional neural networks solve personalized tutoring better understand overfitting control 
 
chunk_size = 15
chunk_overlap = 7

chunk 1 => machine learnin
chunk 2 => learning uses s
chunk 3 =>  uses support v
chunk 4 => pport vector ma
chunk 5 => ctor machines s
chunk 6 => hines support s
'''
chunk_size = 200
chunk_overlap = 20
for index in range(0,len(data),chunk_size-chunk_overlap):
    print(data[index:index+chunk_size])

rag pipeline 
 step 1 
 datum loaderwordpdftxthow   tool use langchainframeworkinterview questionshow extract text information pdf text langchainpdf image inside image text contentsolution   ocr model
solution   ocr modelpaddle ocrgraph base imagesolution   vlmocr help paddle ocr model extract text inside imagevlm summary description graphhow handle pdf have text image content hybrid1 text   langch
ybrid1 text   langchain2 image   vlm ocr depend scenariomerge text langchain document loader second vlm ocr beforesplitte document multiple part 
 step 2 
 text normalizati 
 1page number 
 2unwante c
 number 
 2unwante charactersremove emojisspl characterspunctuation 
 3exapand word example   notexample 2 soon possible 
 4correcte wordsnot possible correct word 


 10 
 nlp 
 1nltknatural language
nltknatural language toolkit 
 2spacy 
 step 3 
 chunk   convert document multiple part 
   chunk 
 tool   langchain 
 step 4 
 embedding   convert chunk numbersvector 
 tool langchain 
 technique

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap = 100,
    separators=['\n\n','.','!','\n',' ','']
)

chunks = splitter.create_documents([data])
# for chunk in chunks:
#     print(chunk.page_content)
chunks

[Document(metadata={}, page_content='rag pipeline \n step 1'),
 Document(metadata={}, page_content='datum loaderwordpdftxthow   tool use langchainframeworkinterview questionshow extract text information pdf text langchainpdf image inside image text contentsolution   ocr modelpaddle ocrgraph base'),
 Document(metadata={}, page_content='pdf text langchainpdf image inside image text contentsolution   ocr modelpaddle ocrgraph base imagesolution   vlmocr help paddle ocr model extract text inside imagevlm summary description graphhow'),
 Document(metadata={}, page_content='vlmocr help paddle ocr model extract text inside imagevlm summary description graphhow handle pdf have text image content hybrid1 text   langchain2 image   vlm ocr depend scenariomerge text'),
 Document(metadata={}, page_content='pdf have text image content hybrid1 text   langchain2 image   vlm ocr depend scenariomerge text langchain document loader second vlm ocr beforesplitte document multiple part'),
 Document(metadata=

In [15]:
chunks[0].metadata={'file_name':'data.txt'}
chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='rag pipeline \n step 1'),
 Document(metadata={}, page_content='datum loaderwordpdftxthow   tool use langchainframeworkinterview questionshow extract text information pdf text langchainpdf image inside image text contentsolution   ocr modelpaddle ocrgraph base'),
 Document(metadata={}, page_content='pdf text langchainpdf image inside image text contentsolution   ocr modelpaddle ocrgraph base imagesolution   vlmocr help paddle ocr model extract text inside imagevlm summary description graphhow'),
 Document(metadata={}, page_content='vlmocr help paddle ocr model extract text inside imagevlm summary description graphhow handle pdf have text image content hybrid1 text   langchain2 image   vlm ocr depend scenariomerge text'),
 Document(metadata={}, page_content='pdf have text image content hybrid1 text   langchain2 image   vlm ocr depend scenariomerge text langchain document loader second vlm ocr beforesplitte document multiple part

### Chunk Embeddings (Converts chunks to vectors)

In [16]:
chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='rag pipeline \n step 1'),
 Document(metadata={}, page_content='datum loaderwordpdftxthow   tool use langchainframeworkinterview questionshow extract text information pdf text langchainpdf image inside image text contentsolution   ocr modelpaddle ocrgraph base'),
 Document(metadata={}, page_content='pdf text langchainpdf image inside image text contentsolution   ocr modelpaddle ocrgraph base imagesolution   vlmocr help paddle ocr model extract text inside imagevlm summary description graphhow'),
 Document(metadata={}, page_content='vlmocr help paddle ocr model extract text inside imagevlm summary description graphhow handle pdf have text image content hybrid1 text   langchain2 image   vlm ocr depend scenariomerge text'),
 Document(metadata={}, page_content='pdf have text image content hybrid1 text   langchain2 image   vlm ocr depend scenariomerge text langchain document loader second vlm ocr beforesplitte document multiple part

In [17]:
# pip install sentence-transformers

In [18]:
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-miniLM-L6-V2' #Encoder only transformer architecture
)
embedding_model

c:\Users\graj5\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2408.81it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-miniLM-L6-V2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [19]:
# chunks

# chunks into vectors

In [20]:
# pip install faiss-cpu 

In [21]:
vectordb = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
vectordb

In [22]:
user_query = 'What is Machine Learning ?'
r_chunks = vectordb.similarity_search(user_query) #Euclidean Distance
r_chunks 

[Document(id='1d781db4-0658-4882-ae44-060c203296a6', metadata={}, page_content='retrieval extract relevant document vector database \n technique   similarity search \n statistic concepts1cosine similarity2euclidean distanceknninterview question \n rag use algorithm retrieval'),
 Document(id='0524ec00-0bd1-47a1-8263-cad578321a86', metadata={}, page_content='rag use algorithm retrieval \n knn algorithmeuclidean distance formulastep 10 \n llm large language model \n tool langchain'),
 Document(id='9909ea37-1f54-458f-b4a4-76b8333fbd91', metadata={}, page_content='technique   transformersencoder architecture \n step 5 \n vector db \n tool langchain \n db faissoct 10   1248 pm \n rag pipeline \n step 1'),
 Document(id='853048f6-ff23-4040-b355-b6536b04200b', metadata={}, page_content='llm large language model \n tool langchain \n use structure outputdiagramdocument text normalizationchunkingembeddingsvectordbloaderuser querytokenizationquery embeddingsretrievalrelevant document')]

In [23]:
# 1)
retrieval_text = ''
for chunk in r_chunks:
    retrieval_text+=chunk.page_content
# print(retrieval_text)
# 2)
retrieval = [chunk.page_content for chunk in r_chunks]
retrieval_text = ' '.join(retrieval)

In [24]:
updated_r_chunks = set()

for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)

R_text ='\n'.join(updated_r_chunks)
R_text

'retrieval extract relevant document vector database \n technique   similarity search \n statistic concepts1cosine similarity2euclidean distanceknninterview question \n rag use algorithm retrieval\nrag use algorithm retrieval \n knn algorithmeuclidean distance formulastep 10 \n llm large language model \n tool langchain\ntechnique   transformersencoder architecture \n step 5 \n vector db \n tool langchain \n db faissoct 10   1248 pm \n rag pipeline \n step 1\nllm large language model \n tool langchain \n use structure outputdiagramdocument text normalizationchunkingembeddingsvectordbloaderuser querytokenizationquery embeddingsretrievalrelevant document'

In [25]:
def retrieval_text(query:str,k=3) -> str:
    R_chunks = vectordb.similarity_search(query,k=k)
    R_chunks = {doc.page_content for doc in R_chunks}
    R_Text = '\n'.join(R_chunks)
    return [R_Text]

try:
    r_text = retrieval_text('what is machine learning',4)
except Exception as e:
    print(f'Error : {e}')



In [26]:
model_ollama = ChatOllama(
    model='qwen2.5:1.5b',
    temperature=0
)
model_gemini  = ChatGoogleGenerativeAI(
    model = 'gemini-3.5-flash',
    api_key = os.environ['Gemini-Api'],
    temperature=0
)

In [ ]:
def generate_text(query,answer):
    prompt = f'''
    You're a helpful assistant and structure the given context.
    input : {query}
    output : {answer}
    
    Note:
    1)Don't add any extra contents just structure my given output.
    2)If the query is not related to context just say 'query is not related to context'.
    Output Format:  
    input : {query}
    output : structured output    
    '''
    # try:
    #     response = model_gemini.invoke(prompt)
    #     return response.content[0]['text']
    # except:
        # response = model_ollama.invoke(prompt)
        # return response.content
    response = model_ollama.invoke(prompt)
    return response.content
retrieval_t = retrieval_text(user_query,2)
final_response = generate_text(user_query,retrieval_t)
print(final_response)

In [ ]:
def r_search(query,k=3):
    R_chunks = vectordb.similarity_search(query,k=k)
    # print(R_chunks)
    R_chunks = {doc.page_content for doc in R_chunks}
    R_Text = '\n'.join(R_chunks)
    return R_Text
def g_text(r_search,query):
        import os
        prompt = f'''
                    You're a helpful assistant
                    Assigned Task for you : Structure my output => {r_search}
                    for this input = > {query}
                    Note : 
                    1) Don't add extra contents just structure mentioned output.
                    2) If there is mistake in output correct or else keep the original output
                    with structured result.
                    Output structure:
                    Input : {query}
                    Output : structured output
            '''
        llm_model = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash", #gemini-3.5-flash,gemini-3.6-flash,gemini-2.5-pro
            api_key=os.environ['GEMINI_API_KEY'],
            temperature = 0.0
        )
        response = llm_model.invoke(prompt).content
        return response
user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)
r_response = r_search(user_prompt)
g_response = g_text(r_response,user_prompt)
print(g_response)

Input : Explain Machine Learning
Output : Machine learning uses backpropagation and can apply policy gradient to help a model generalize effectively for medical diagnosis.


In [ ]:
len('''machine learning uses support vector machines support supply chain optimization improving loss functions modern systems ''')

120

In [ ]:
len(''' sentiment analysis deep learning benefits k nearest neighbors strengthens activation functions improves decision quality  ''')

123

In [ ]:
len('''  neural networks applies naive bayes document summarization data augmentation helps model generalize effectively 
 practical study natural language processing shows principal component analysis improve video understanding refining class imbalance training ''')

257

In [ ]:
len(''' eural networks applies naive bayes document summarization data augmentation helps model generalize effectively ''')

112

In [ ]:
len(''' practical study natural language processing shows principal component analysis improve video understanding refining class imbalance training  ''')

143

In [ ]:
'''  neural networks applies naive bayes document summarization data augmentation helps model generalize effectively 
 practical study natural language processing shows principal component analysis improve video understanding refining class imbalance training 
 
 
 researchers use computer vision convolutional neural networks solve personalized tutoring better understand overfitting control '''.split('\n')

['  neural networks applies naive bayes document summarization data augmentation helps model generalize effectively ',
 ' practical study natural language processing shows principal component analysis improve video understanding refining class imbalance training ',
 '',
 '',
 ' researchers use computer vision convolutional neural networks solve personalized tutoring better understand overfitting control ']